# 06 - Semantic chunking

Semantic chunking tries to split text when the topic changes. Unlike fixed-size, sentence, or paragraph chunking, the goal is not just a shape boundary. The goal is meaning.

Production semantic chunking usually uses embeddings. This notebook uses a transparent keyword-overlap heuristic so learners can see the idea without downloading a model.

In [ ]:
from collections import Counter
from pathlib import Path
import re

from pypdf import PdfReader

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/sample_documents/AUG242026_05B2203_AAO.pdf").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the Awesome_RAG checkout")
PDF_PATH = ROOT / "data" / "sample_documents" / "AUG242026_05B2203_AAO.pdf"

def normalize_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def load_pdf_pages(path):
    reader = PdfReader(str(path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = normalize_text(page.extract_text() or "")
        if text:
            pages.append({"page": page_number, "text": text})
    return pages

def chunk_stats(chunks):
    lengths = [len(chunk["text"].split()) for chunk in chunks]
    return {"chunks": len(chunks), "min_words": min(lengths), "avg_words": round(sum(lengths) / len(lengths), 1), "max_words": max(lengths)}

def print_chunks(chunks, limit=5, preview_chars=650):
    for chunk in chunks[:limit]:
        print(f"{chunk['id']} | page {chunk['page_start']}-{chunk['page_end']}")
        print(chunk["text"][:preview_chars])
        print("-" * 100)
    if len(chunks) > limit:
        print(f"... {len(chunks) - limit} more chunks")

pages = load_pdf_pages(PDF_PATH)
{"pages": len(pages), "words": sum(len(page["text"].split()) for page in pages)}

## Semantic chunking function

This teaching version uses keyword overlap:

- Split the page into sentences.
- Convert each sentence into content terms by removing small stopwords.
- Keep adding sentences while the topic appears related.
- Start a new chunk when shared terms become low and the current chunk is large enough.

This is not as strong as embedding-based semantic chunking, but it makes the decision process visible.

In [ ]:
def split_sentences(text):
    return [part.strip() for part in re.split(r"(?<=[.!?])\s+", text) if part.strip()]


def content_terms(text):
    stopwords = {
        "the", "and", "or", "a", "an", "to", "of", "in", "for", "on", "by", "with",
        "is", "are", "was", "were", "be", "this", "that", "from", "as", "at", "it",
    }
    return {word for word in re.findall(r"[a-z0-9]+", text.lower()) if word not in stopwords and len(word) > 2}


def semantic_chunks(pages, max_words=220, min_shared_terms=2, min_words=40):
    """Create chunks by splitting when neighboring sentence topics appear to change."""
    chunks = []

    for page in pages:
        current = []
        current_terms = set()
        current_words = 0

        for sentence in split_sentences(page["text"]):
            terms = content_terms(sentence)
            count = len(sentence.split())

            topic_shift = current and current_words >= min_words and len(current_terms & terms) < min_shared_terms
            too_large = current and current_words + count > max_words

            if topic_shift or too_large:
                chunks.append({
                    "id": f"semantic-{len(chunks) + 1}",
                    "page_start": page["page"],
                    "page_end": page["page"],
                    "text": " ".join(current),
                })
                current = []
                current_terms = set()
                current_words = 0

            current.append(sentence)
            current_terms.update(terms)
            current_words += count

        if current:
            chunks.append({
                "id": f"semantic-{len(chunks) + 1}",
                "page_start": page["page"],
                "page_end": page["page"],
                "text": " ".join(current),
            })

    return chunks

chunks = semantic_chunks(pages, max_words=220, min_shared_terms=2, min_words=40)
chunk_stats(chunks)

In [ ]:
print_chunks(chunks, limit=6)

all_terms = Counter(term for chunk in chunks for term in content_terms(chunk["text"]))
all_terms.most_common(12)

## How to judge it

Semantic chunking is useful when documents mix topics without clear headings. It is more complex than paragraph or sentence chunking, so it should earn its place with better retrieval results.

For production RAG, replace this keyword-overlap heuristic with embedding similarity or a document parser that exposes semantic sections.